# Week 3 Day 2: CatBoost, in Colab

In today's lesson you fitted XGBoost on the client's late deliveries. This notebook fits a third boosting library, **CatBoost**, on the same rows and the same columns, and puts it on the same scorecard. It takes about 25 minutes.

**Why Colab.** The lesson's code runs in your browser. CatBoost has no version that runs there, so it runs here instead, on a computer Google lends you. Press **Runtime → Run all**, or run the cells one by one with Shift+Enter. If you have no Google account, you can still read this notebook on GitHub with every output already printed.

**What CatBoost does differently.** The name is short for "categorical boosting". You hand it the category columns as plain text (`Eastport`, `express`, `C189`) and it turns them into numbers itself. There is no one-hot step and no `TargetEncoder`. The way it does this is built to avoid the leak you met in target encoding, and a cell below explains how.

**The rules are the lesson's.** The same training rows, the same bar, a cut-off chosen out-of-fold on the training rows only, and each model scored once on the comparison year. The comparison year is a comparison, not a fresh test: the course's builder looked at it while planning the day. October 2026 is the month nobody has seen, and it is not in these files.

In [1]:
%pip install -q catboost==1.2.10

import catboost, pandas, sklearn, numpy
print("catboost", catboost.__version__)
print("pandas", pandas.__version__)
print("scikit-learn", sklearn.__version__)
print("numpy", numpy.__version__)

Note: you may need to restart the kernel to use updated packages.


catboost 1.2.10
pandas 3.0.2
scikit-learn 1.8.0
numpy 2.5.5


The first line installs CatBoost 1.2.10, the version this notebook was written for. Nothing else is installed: Colab already has pandas and scikit-learn. `%pip` always prints a note that you may need to restart; CatBoost was not loaded before, so you do not.

The saved run printed catboost 1.2.10, pandas 3.0.2 and scikit-learn 1.8.0, the same pandas and scikit-learn as the lesson's page. Colab's own versions may be older or newer, so your figures can differ from the ones quoted below in the third decimal.

In [2]:
import os
import math
import numpy as np
import pandas as pd

# Set the environment variable DATA_DIR to a folder holding the two files to read them from there instead of GitHub.
DATA_DIR = os.environ.get("DATA_DIR")
BASE = "https://raw.githubusercontent.com/talhaturab/fde-course-notebooks/main/data/"

def data_file(name):
    return os.path.join(DATA_DIR, name) if DATA_DIR else BASE + name

history = pd.read_csv(data_file("shipments_history.csv"))
accounts = pd.read_csv(data_file("customers_history.csv"))
known = history.merge(accounts[["customer_id", "segment"]], on="customer_id", how="left")
known = known.sort_values(["dispatched_at", "shipment_id"]).reset_index(drop=True)
known["promised"] = known["service_level"].map({"express": 2, "standard": 4})
known["late"] = (known["delivery_days"] > known["promised"]).astype(int)

print(len(known), "delivered shipments from", known["dispatched_at"].min(), "to", known["dispatched_at"].max())
print(len(accounts), "accounts")
known.head(3)

21511 delivered shipments from 2024-01-01 10:43 to 2026-09-30 19:22
240 accounts


,shipment_id,customer_id,origin_hub,destination_region,service_level,dispatched_at,distance_km,weight_kg,volume_l,delivery_days,segment,promised,late
0,H-100001,C189,Eastport,West,standard,2024-01-01 10:43,369,4.4,21.2,1.2,Retail,4,0
1,H-100002,C007,Eastport,East,express,2024-01-01 11:13,191,5.1,17.5,0.6,E-commerce,2,0
2,H-100003,C006,Northgate,Central,standard,2024-01-01 11:26,508,8.9,NaN,3.4,E-commerce,4,0


This is the lesson's setup, line for line. `known` is the client's full history (21,511 delivered shipments, January 2024 to September 2026) with each account's `segment` joined on from the accounts file, sorted by dispatch time. A delivery is **late** when it took more days than promised: 2 for express, 4 for standard.

The table shows the first three shipments. `customer_id` is the account (240 in the file), and `delivery_days` is the answer the model learns from.

In [3]:
when = pd.to_datetime(known["dispatched_at"])
known["weekday"] = when.dt.weekday   # 0 is Monday
known["hour"] = when.dt.hour
known["month"] = when.dt.month
known["day"] = when.dt.day
known["litres_per_kg"] = known["volume_l"] / known["weight_kg"]

categories = ["service_level", "origin_hub", "destination_region", "segment", "customer_id"]
numbers = ["distance_km", "weight_kg", "volume_l", "litres_per_kg", "weekday", "hour", "month", "day"]
columns = numbers + categories

train = known[known["dispatched_at"] < "2025-10-01"].reset_index(drop=True)
comparison = known[known["dispatched_at"] >= "2025-10-01"].reset_index(drop=True)
need = math.ceil(2 / 3 * comparison["late"].sum())

print("training rows:", len(train), "of which late:", train["late"].sum())
print("comparison year:", len(comparison), "of which late:", comparison["late"].sum())
print("the bar: catch at least", need, "of them, with accuracy above 0.808")

training rows: 11645 of which late: 2587
comparison year: 9866 of which late: 2362
the bar: catch at least 1575 of them, with accuracy above 0.808


These are today's columns, made the lesson's way: the four date parts from `dispatched_at`, and `litres_per_kg` for the bulky, light parcels that go to manual sorting. The two counts (hub load and the account's recent shipments) are left out, as in the lesson, because out-of-fold they added nothing we could measure.

The saved run printed:
- **training rows** (dispatched January 2024 to September 2025): 11,645, of which 2,587 late
- **the comparison year** (October 2025 to September 2026): 9,866, of which 2,362 late
- **the bar**: catch at least 1,575 of the 2,362 (two in three, rounded up), with accuracy above 0.808, the distance rule's

Some weights and volumes are blank, so some `litres_per_kg` are blank too. CatBoost needs no `SimpleImputer`: at every question a tree asks, it sends the blanks to the low side, as if they were smaller than any number.

In [4]:
from catboost import CatBoostClassifier, CatBoostRegressor

SETTINGS = dict(iterations=2000, learning_rate=0.05, od_type="Iter", od_wait=50,
                use_best_model=True, random_seed=0, verbose=False,
                allow_writing_files=False)  # no catboost_info folder of log files

def fit_stopped(model, X, y):
    # the last 15% of the rows (by time) is the stopping slice; the rest is what the trees learn from
    cut = int(len(X) * 0.85)
    model.fit(X.iloc[:cut], y.iloc[:cut], eval_set=(X.iloc[cut:], y.iloc[cut:]))
    return model

def late_model(cats):
    return CatBoostClassifier(eval_metric="Accuracy", cat_features=cats, **SETTINGS)

model = fit_stopped(late_model(categories), train[columns], train["late"])
print("learned from", int(len(train) * 0.85), "rows; stopping slice", len(train) - int(len(train) * 0.85), "rows")
print("trees kept:", model.tree_count_, "of up to 2000")

learned from 9898 rows; stopping slice 1747 rows
trees kept: 206 of up to 2000


This is the lesson's early stopping, in CatBoost's words:
- `iterations=2000` is the most trees it may grow, and `learning_rate=0.05` is the lesson's small weight for each tree.
- `eval_set` is the stopping slice: the last 15% of the training rows by time. CatBoost scores it after every tree.
- `eval_metric="Accuracy"` scores the slice by the share right, the scorecard's own measure.
- `od_type="Iter"` with `od_wait=50` stops when 50 trees in a row have not beaten the best score so far ("od" is short for overfitting detector).
- `use_best_model=True` then throws away the trees after the best one, like reading `best_iteration` in XGBoost.
- `cat_features` names the text columns. Nothing else is done to them.

The saved run learned from 9,898 rows, stopped on the last 1,747, and **kept 206 trees**: the slice was scored best after tree 206, and the 50 trees after it did no better.

CatBoost's trees are 6 levels deep by default (XGBoost's in the lesson were 4). Each CatBoost tree also asks the same question at every node of a level, so its trees are simpler than their depth suggests.

In [5]:
from sklearn.model_selection import TimeSeriesSplit

def out_of_fold(make_model, X, y):
    # Week 2's time-ordered rounds: fit on each round's earlier rows, predict its later rows
    shares, actual, trees = [], [], []
    for fit_rows, held_rows in TimeSeriesSplit(4).split(X):
        m = fit_stopped(make_model(), X.iloc[fit_rows], y.iloc[fit_rows])
        shares.append(m.predict_proba(X.iloc[held_rows])[:, 1])
        actual.append(y.iloc[held_rows].to_numpy())
        trees.append(m.tree_count_)
    return np.concatenate(shares), np.concatenate(actual), trees

def choose_cut(share, actual):
    rows = []
    for cut in np.round(np.arange(0.05, 0.96, 0.05), 2):
        rows.append({"cut": cut, "correct": int(((share >= cut) == actual).sum()),
                     "caught": int(((share >= cut) & (actual == 1)).sum())})
    table = pd.DataFrame(rows)
    need_oof = math.ceil(2 / 3 * actual.sum())
    keep = table[table["caught"] >= need_oof].sort_values(["correct", "caught"], ascending=False)
    print(len(actual), "out-of-fold rows,", actual.sum(), "late; a cut-off must catch at least", need_oof)
    print(keep.head(3).to_string(index=False))
    return float(keep.iloc[0]["cut"])

share, actual, trees = out_of_fold(lambda: late_model(categories), train[columns], train["late"])
print("trees kept in each round:", trees)
cut = choose_cut(share, actual)
print("chosen cut-off:", cut)

trees kept in each round: [12, 164, 135, 166]
9316 out-of-fold rows, 2132 late; a cut-off must catch at least 1422
 cut  correct  caught
0.40     8510    1608
0.35     8509    1685
0.45     8499    1530
chosen cut-off: 0.4


The cut-off is chosen by Day 1's rule, on the training rows only. `out_of_fold` runs Week 2's four time-ordered rounds. Each round fits a fresh model (with its own stopping slice, the last 15% of that round's rows) and predicts the round's later rows. Every training row after the first fifth gets a share of "late" from a model that never saw it.

`choose_cut` then tries 0.05, 0.10, …, 0.95, keeps the cut-offs that catch at least two in three of the out-of-fold late deliveries, and takes the one with the most correct guesses. On a tie it takes the one that catches more.

The saved run printed:
- 9,316 out-of-fold rows, 2,132 of them late, so a cut-off must catch at least 1,422
- **cut-off 0.40**: 8,510 correct (0.913), catching 1,608
- 0.35 came second, one correct guess behind (8,509), catching 1,685

The trees kept in each round were 12, 164, 135 and 166. The first round is given only about 2,300 rows: it learns from about 1,980 and stops on the last 350 or so, where the share right jumps around, so it stops early.

A one-guess lead is a coin toss, so on Colab the rule may pick 0.35 instead. The next cell uses whatever `cut` this cell chose.

In [6]:
def score_once(model, X, y, cut):
    late = model.predict_proba(X)[:, 1] >= cut
    accuracy = (late == y).mean()
    caught = int((late & (y == 1)).sum())
    clears = bool(caught >= need and accuracy > 0.808)
    print(f"cut-off {cut}: accuracy {accuracy:.3f}, caught {caught} of {int(y.sum())}, clears the bar: {clears}")

score_once(model, comparison[columns], comparison["late"], cut)

cut-off 0.4: accuracy 0.911, caught 1764 of 2362, clears the bar: True


This is the model from the first fit (all the training rows, 206 trees) with the cut-off chosen out-of-fold, scored once on the comparison year.

The saved run printed **accuracy 0.911, caught 1,764 of 2,362: it clears the bar** (1,575 caught, accuracy above 0.808).

For comparison, the lesson's declared model, XGBoost with the same columns at its out-of-fold cut-off 0.4, scored 0.909 and caught 1,727. The forest with the same columns scored 0.902 and caught 1,816. The gap to XGBoost is 0.002 in accuracy and 37 deliveries caught, on one draw of the data (this notebook ran CatBoost on the committed draw only). A gap that small, on one draw, is not a ranking. Day 3 compares models properly.

What CatBoost did differently is how it handled the text columns. It needed no encoder.

## Why CatBoost needs no target encoder

In the lesson, target encoding replaced each account with its late share, and the leak was that a row could help write its own number. With one-row accounts, the "late share" *was* the answer.

CatBoost builds the same kind of number, a late share for each value of a category column, but it builds it in a way that never lets a row see its own answer. Its makers call this **ordered target statistics**. In plain words:
1. It puts the training rows in a random order.
2. For each row, it works out the account's late share from **only the rows before it** in that order. A small starting value is mixed in, so an account with few rows before it gets a cautious number (a smoothing, like `TargetEncoder`'s).
3. The first row of an account has no rows before it, so it gets the starting value alone. A row's own late or on-time never enters its own number.
4. It repeats this with a few different orders, so no row is always first.

For new rows, like the comparison year, it uses every training row of that account, because their answers really are known by then.

`TargetEncoder` avoided the leak by cross-fitting (each row encoded from the other folds). CatBoost avoids it by ordering. Either way, the rule is today's: a column may hold only what you could have known without the row's own answer.

In [7]:
no_account = [c for c in columns if c != "customer_id"]
no_account_cats = [c for c in categories if c != "customer_id"]

share2, actual2, trees2 = out_of_fold(lambda: late_model(no_account_cats), train[no_account], train["late"])
cut2 = choose_cut(share2, actual2)
model2 = fit_stopped(late_model(no_account_cats), train[no_account], train["late"])
print("chosen cut-off:", cut2, "; trees kept:", model2.tree_count_)
score_once(model2, comparison[no_account], comparison["late"], cut2)

9316 out-of-fold rows, 2132 late; a cut-off must catch at least 1422
 cut  correct  caught
0.45     8482    1523
0.40     8471    1581
0.35     8467    1649


chosen cut-off: 0.45 ; trees kept: 205
cut-off 0.45: accuracy 0.906, caught 1660 of 2362, clears the bar: True


This is the same recipe with `customer_id` dropped, to see what the account column is worth. The cut-off is chosen again, out-of-fold, because a different model can need a different cut-off.

The saved run printed:
- **cut-off 0.45**: 8,482 correct out-of-fold (0.910), catching 1,523. With the account the best was 8,510 correct, catching 1,608 (at 0.40).
- on the comparison year: **accuracy 0.906, caught 1,660**. It still clears the bar, but with the account the figures were 0.911 and 1,764.

So the account column is worth about 28 more correct guesses out-of-fold, and a cut-off that catches more. The lesson found the same with `TargetEncoder`: some accounts' parcels wait longer, and no other column says which.

In [8]:
def days_model():
    return CatBoostRegressor(loss_function="MAE", cat_features=categories, **SETTINGS)

days = fit_stopped(days_model(), train[columns], train["delivery_days"])
guess = days.predict(comparison[columns])
miss = np.abs(guess - comparison["delivery_days"]).mean()

late = guess > comparison["promised"]
accuracy = (late == comparison["late"]).mean()
caught = int((late & (comparison["late"] == 1)).sum())
print("trees kept:", days.tree_count_)
print(f"average miss on the comparison year: {miss:.3f} days")
print(f"days over the promise: accuracy {accuracy:.3f}, caught {caught} of {comparison['late'].sum()}, "
      f"clears the bar: {bool(caught >= need and accuracy > 0.808)}")

trees kept: 680
average miss on the comparison year: 0.485 days
days over the promise: accuracy 0.917, caught 1789 of 2362, clears the bar: True


This model predicts **how many days** a delivery takes, instead of late or not. `loss_function="MAE"` is CatBoost's code for the average miss in days, the same thing XGBoost's absolute error aimed at in the lesson (Day 3 explains the name). Aiming at the average miss makes each leaf answer with a median, as Day 1's `absolute_error` did. The stopping slice is scored the same way.

Then the days are read against the promise: a delivery is called late when the predicted days are more than the 2 or 4 promised. No cut-off is chosen, so nothing is chosen on the comparison year.

The saved run printed:
- trees kept: 680
- **average miss: 0.485 days**
- **days over the promise: accuracy 0.917, caught 1,789 of 2,362: it clears the bar**

The lesson's XGBoost days model, with the same columns, missed by 0.484 days on average, and read against the promise it scored 0.915 and caught 1,744. The two average misses are a thousandth of a day apart, which is no real difference. Both are well below the line's 0.682 from Week 2.

In [9]:
importance = pd.Series(model.get_feature_importance(), index=model.feature_names_)
importance.sort_values(ascending=False).round(1)

distance_km           40.5
weight_kg             13.6
weekday                8.4
origin_hub             7.7
customer_id            7.2
service_level          5.9
litres_per_kg          4.1
segment                3.4
hour                   2.6
volume_l               2.6
month                  2.4
destination_region     1.2
day                    0.4
dtype: float64

`get_feature_importance()` says how much each column changed the late-or-not model's answers, on average, as shares that add up to 100.

The saved run put `distance_km` far ahead (40.5), then `weight_kg` (13.6), `weekday` (8.4), `origin_hub` (7.7) and `customer_id` (7.2). `day` of the month came last (0.4).

Read it as a rough list only. It does not say in which direction a column pushes, or for which deliveries, and two columns that carry the same thing can split the credit. Day 4's SHAP does this properly, one delivery at a time.

## Exercises

Each one changes one thing and reads the out-of-fold figures, never the comparison year. Use `out_of_fold` and `choose_cut` from above, or read the share right at 0.4 directly with `((share >= 0.4) == actual).mean()`.

1. **The learning rate.** Copy `SETTINGS`, set `learning_rate` to 0.2, and fit the late-or-not model on the training rows. How many trees does it keep, against 206 at 0.05? Why?
2. **The hub count.** Add a column `hub_load_7d`: how many shipments left the same hub in the 7 days before this one (only earlier rows, never this one or later). Run it out-of-fold with today's columns. Does it help, and what does the lesson say about testing it?
3. **Depth.** CatBoost's trees are 6 levels deep by default. Try `depth=4` and `depth=8` (pass `depth=...` to `CatBoostClassifier`). Compare the correct guesses out-of-fold at the cut-off 0.4.

<details>
<summary><b>Answers</b> (click to open)</summary>

These figures come from a run with the same versions as the saved run.

**1. The learning rate.** At 0.2 the model keeps **74 trees**, against 206. Each tree takes a bigger step, so fewer are needed before the stopping slice stops improving. Out-of-fold at 0.4 it got 8,568 right (0.920) and caught 1,654, a little better than 0.05's 8,510 and 1,608. That does not make 0.2 the right choice. The learning rate was declared before any run; trying settings and keeping the best is a tuning search, which is Day 3's topic and needs its own rule.

```python
fast = dict(SETTINGS, learning_rate=0.2)
quick = fit_stopped(CatBoostClassifier(eval_metric="Accuracy", cat_features=categories, **fast),
                    train[columns], train["late"])
print(quick.tree_count_)
```

**2. The hub count.** Out-of-fold at 0.4 it got 8,528 right (0.915) and caught 1,593, against 8,510 and 1,608 without it: 18 more right, 15 fewer caught, which is close to nothing. That is the lesson's finding. No out-of-fold round learns from one busy season and is tested on the next, so the rounds cannot test a column that matters in busy weeks. Do not look at the comparison year to decide: keeping a column because it scored well there would be choosing on the comparison year.

```python
def earlier_7d(times):
    t = times.to_numpy()   # in dispatch order within each hub
    return np.arange(len(t)) - np.searchsorted(t, t - np.timedelta64(7, "D"))

known["when"] = pd.to_datetime(known["dispatched_at"])
known["hub_load_7d"] = known.groupby("origin_hub")["when"].transform(earlier_7d)
train = known[known["dispatched_at"] < "2025-10-01"].reset_index(drop=True)
with_hub = columns + ["hub_load_7d"]
s, a, _ = out_of_fold(lambda: late_model(categories), train[with_hub], train["late"])
print(((s >= 0.4) == a).sum(), ((s >= 0.4) & (a == 1)).sum())
```

**3. Depth.** At depth 4 it got 8,510 right (0.913) and caught 1,599; at depth 8, 8,569 (0.920) and 1,635. The default depth 6 got 8,510 and 1,608. Deeper trees did a little better here. As in exercise 1, that is a hint for Day 3's tuning, not a reason to change the declared model today.

</details>

## What you did

- You fitted CatBoost on today's columns with the category columns as plain text, early stopping on the last 15% of the training rows, and a cut-off chosen out-of-fold.
- Scored once, it cleared the bar on the comparison year, level with the lesson's XGBoost.
- You saw why it needs no target encoder: each row's account number is built only from rows before it in a random order.
- Without the account column it still cleared the bar, but caught fewer.
- For days it missed by 0.485 days on average, against XGBoost's 0.484.

Back to the lesson for the scorecard and what to tell the client.